In [1]:
// Your occupied coordinates
val occupiedGrid: Set<Pair<Int, Int>> = setOf(
    0 to 0,
    1 to 0,
    1 to 1,
    0 to 1,
    -1 to 1,
    -1 to 0 // 4 is occupied too
)

In [4]:
private val nonAlphaRegex = Regex("[^A-Za-z0-9$]")

private val vdmReservedWords = setOf("module")

fun toVdmName(name: String): String {
    val vdmName = name.replace(' ', '$').replace('_', '$').replace(nonAlphaRegex, "")
    return if (Character.isDigit(vdmName.first()) || vdmName in vdmReservedWords) {
        "\$$vdmName"
    } else {
        vdmName
    }
}

println(toVdmName("xyz.abc"))

xyzabc


In [6]:
/**
 * Lazily generates an infinite stream of (X, Y) coordinates winding outward
 * as a spiral from (0, 0).
 */
fun spiralSequence(step: Int): Sequence<Pair<Int, Int>> = sequence {
    var x = 0
    var y = 0
    var dx = 0
    var dy = step

    while (true) {
        yield(x to y)
        // Change direction when we hit a corner of the spiral boundary
        if (x == y || (x < 0 && x == -y) || (x > 0 && x == 1 - y)) {
            val temp = dx
            dx = -dy
            dy = temp
        }
        x += dx
        y += dy
    }
}

In [9]:
// Find the first vacant coordinate
val fresh2DSpace = spiralSequence(100).first { it !in occupiedGrid }
// Result: (-1 to -1) -> the next open coordinate in the 2D spiral

In [12]:
fun freshNames(from: Set<String> = emptySet(), prefix: String = "a", start: Int = from.size) : Sequence<String> =
    generateSequence(prefix + start) { index ->
        val suffix = index.substringAfterLast(prefix).toIntOrNull() ?: start
        "$prefix${suffix + 1}"
    }

In [14]:
println(freshNames().take(10).toList())
println(freshNames().take(10).flatten())
println(freshNames(setOf("a1, a5, a10, aircraft1")).take(10).toList())


[a0, a1, a2, a3, a4, a5, a6, a7, a8, a9]
[a1, a2, a3, a4, a5, a6, a7, a8, a9, a10]


In [11]:
println(occupiedGrid)
println(fresh2DSpace)
println(spiralSequence(100).take(10).toList())

[(0, 0), (1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0)]
(-100, 0)
[(0, 0), (-100, 0), (-200, 0), (-300, 0), (-400, 0), (-500, 0), (-600, 0), (-700, 0), (-800, 0), (-900, 0)]


In [5]:
enum class Day {
    MONDAY, TUESDAY, WEDNESDAY, THURSDAY, FRIDAY
}

// 1. Define the extension operators
operator fun <T> Array<T>.get(day: Day): T = this[day.ordinal]
operator fun <T> Array<T>.set(day: Day, value: T) {
    this[day.ordinal] = value
}

fun f() {
    // 2. Initialize a standard array of the correct size
    val schedule = Array(Day.entries.size) { "Rest" }
    println(schedule.joinToString(separator = ", "))

    // 3. Index the array directly using your Enum values!
    schedule[Day.MONDAY] = "Coding Kotlin"
    schedule[Day.FRIDAY] = "Deploying to Prod"

    println(schedule[Day.MONDAY]) // Output: Coding Kotlin
    println(schedule.joinToString(separator = ", "))
}

f()

Rest, Rest, Rest, Rest, Rest
Coding Kotlin
Coding Kotlin, Rest, Rest, Rest, Deploying to Prod


In [10]:
enum class Quadrant { Q1, Q2, Q3, Q4 }
enum class Convergence { CONVERGING, DIVERGING, OVERTAKING }
operator fun <T> Array<Array<T>>.get(row: Quadrant, col: Quadrant): T = this[row.ordinal][col.ordinal]
operator fun <T> Array<Array<T>>.set(row: Quadrant, col: Quadrant, value: T) { this[row.ordinal][col.ordinal] = value }
val schedule = Array(Quadrant.entries.size) { Array(Quadrant.entries.size) { Convergence.DIVERGING } }
schedule[Quadrant.Q1, Quadrant.Q1] = Convergence.CONVERGING
val CONVERGENCE_MATRIX = arrayOf(
    arrayOf(Convergence.CONVERGING, Convergence.CONVERGING, Convergence.OVERTAKING, Convergence.OVERTAKING),
    arrayOf(Convergence.CONVERGING, Convergence.CONVERGING, Convergence.OVERTAKING, Convergence.OVERTAKING),
    arrayOf(Convergence.OVERTAKING, Convergence.OVERTAKING, Convergence.DIVERGING, Convergence.DIVERGING),
    arrayOf(Convergence.OVERTAKING, Convergence.OVERTAKING, Convergence.DIVERGING, Convergence.DIVERGING),
)
println(CONVERGENCE_MATRIX.joinToString(separator = "\n") { it.joinToString(separator = ", ") })
println(CONVERGENCE_MATRIX[Quadrant.Q1, Quadrant.Q1])

CONVERGING, CONVERGING, OVERTAKING, OVERTAKING
CONVERGING, CONVERGING, OVERTAKING, OVERTAKING
OVERTAKING, OVERTAKING, DIVERGING, DIVERGING
OVERTAKING, OVERTAKING, DIVERGING, DIVERGING
CONVERGING


In [9]:
import kotlinx.serialization.json.Json
import kotlinx.serialization.decodeFromString

// 1. Your payload matches a standard JSON Object structure
val jsonString1 = """
    {
      "Point_Alpha": [[1.0, 2.0], [3.0, 4.0] ],
      "Point_Beta":  [[5.0, 6.0], [7.0, 8.0] ]
    }
""".trimIndent()

val jsonString2 = """
    {
      "Point_Alpha": {
         "first": { "first": 1.0, "second": 2.0 },
         "second": { "first": 3.0, "second": 4.0 }
      },
      "Point_Beta": {
         "first": { "first": 5.0, "second": 6.0 },
         "second": { "first": 7.0, "second": 8.0 }
      }
    }
""".trimIndent()

val asList: Map<String, List<List<Double>>> = Json.decodeFromString(jsonString1)
val asPairs: Map<String, Pair<Pair<Double, Double>, Pair<Double, Double>>> = Json.decodeFromString(jsonString2)

println(asList)
println(asPairs)

{Point_Alpha=[[1.0, 2.0], [3.0, 4.0]], Point_Beta=[[5.0, 6.0], [7.0, 8.0]]}
{Point_Alpha=((1.0, 2.0), (3.0, 4.0)), Point_Beta=((5.0, 6.0), (7.0, 8.0))}


In [28]:
import com.fasterxml.jackson.module.kotlin.jacksonObjectMapper

// 1. Your payload matches a standard JSON Object structure
val jsonString1 = """
    {
      "Point_Alpha": [[1.0, 2.0], [3.0, 4.0] ],
      "Point_Beta":  [[5.0, 6.0], [7.0, 8.0] ]
    }
""".trimIndent()

val mapper = jacksonObjectMapper()
val asList: Map<String, List<List<Double>>> = mapper.readValue(jsonString1)
val asPairs: Map<String, Pair<Pair<Double, Double>, Pair<Double, Double>>> = Json.decodeFromString(jsonString2)

println(asList)
println(asPairs)

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[28], line 1, column 12: Unresolved reference: fasterxml
at Cell In[28], line 11, column 14: Unresolved reference: jacksonObjectMapper

In [26]:
val x = mapOf(1 to 2, 3 to 4, 5 to 6, 7 to 4)
val xv = x.values
println(x)
println(xv)

{1=2, 3=4, 5=6, 7=4}
[2, 4, 6, 4]


In [27]:
import kotlinx.serialization.json.Json
import kotlinx.serialization.decodeFromString

data class Position(val x: Double, val y: Double) {
    //TODO Why doesn't inheritance works well with Data classes? equals/hash?
    //TODO should we really have the dubplication `data class Position` vs. `@Module interface Position`?
    constructor(coordinates: Pair<Double, Double>) : this(coordinates.first, coordinates.second)

    fun toPair() = Pair(x, y)
    override fun toString(): String = "P($x, $y)"
}

data class Velocity(val x: Double, val y: Double) {
    constructor(coordinates: Pair<Double, Double>) : this(coordinates.first, coordinates.second)

    fun toPair() = Pair(x, y)
    override fun toString(): String = "V($x, $y)"
}

data class Aircraft(val position: Position, val velocity: Velocity) {
    constructor(coordinates: Pair<Position, Velocity>) : this(coordinates.first, coordinates.second)

    fun toPair() = Pair(position.toPair(), velocity.toPair())
    override fun toString(): String = "A($position, $velocity)"
}

// 1. Your payload matches a standard JSON Object structure
val jsonString1 = """
    {
      "Point_Alpha": [[1.0, 2.0], [3.0, 4.0] ],
      "Point_Beta":  [[1.0, 2.0], [7.0, 8.0] ],
      "Point_Beta": [[1.0, 2.0], [7.0, 10.0] ]
    }
""".trimIndent() // repetition overrides!

val jsonString2 = """
    {
      "Point_Alpha": {
         "position": { "x": 1.0, "y": 2.0 },
         "velocity": { "x": 3.0, "y": 4.0 }
      },
      "Point_Beta": {
         "position": { "x": 5.0, "y": 6.0 },
         "velocity": { "x": 7.0, "y": 8.0 }
      }
    }
""".trimIndent()

// repeated keys are absorbed
val asList: Map<String, List<List<Double>>> = Json.decodeFromString(jsonString1)

val pairMap = asList.mapValues { (key, matrix) ->
    // inner lists must be well formed position/velocity;
    require(matrix.size == 2 && matrix[0].size == 2 && matrix[1].size == 2) {
        "Malformed aircraft data for key '$key'. Expected exact 2x2 matrix for position and velocity x and y coordinates, got: $matrix"
    }
    val (pos, vel) = matrix
    val (x1, y1) = pos
    val (x2, y2) = vel
    (x1 to y1) to (x2 to y2)
}

val x = pairMap.mapValues { v -> v }
println(x)
val positions = pairMap.values.map { it.first }
println(positions)
require(positions.toSet().size == positions.size) { "Duplicate positions found: $positions" }

println(asList)
println(pairMap)

[(1.0, 2.0), (1.0, 2.0)]


java.lang.IllegalArgumentException: Duplicate positions found: [(1.0, 2.0), (1.0, 2.0)]